***Patient Data Analyzer***

### Raw Data Generation

In [1]:
import numpy as np
import pandas as pd

np.random.seed(100)
total_records = 500

client_codes = [f"C{str(j).zfill(4)}" for j in range(1, total_records + 1)]
client_age = np.random.randint(20, 75, size=total_records).astype(float)
client_sex = np.random.choice(["Male", "Female"], size=total_records)
client_zone = np.random.choice(["North", "South", "East", "West"], size=total_records)
client_weight_index = np.random.normal(25.5, 4.0, size=total_records)
client_bp_level = np.random.normal(118, 14, size=total_records)
client_lipid = np.random.normal(195, 28, size=total_records)
client_sugar = np.random.normal(95, 18, size=total_records)

score_formula = (
    -7.5
    + 0.03 * client_age
    + 0.07 * client_weight_index
    + 0.02 * client_bp_level
    + 0.01 * client_lipid
    + 0.03 * client_sugar
)
calc_prob = 1 / (1 + np.exp(-score_formula))
health_target = (calc_prob > np.random.rand(total_records)).astype(int)

raw_df = pd.DataFrame(
    {
        "client_code": client_codes,
        "client_age": client_age,
        "client_sex": client_sex,
        "client_zone": client_zone,
        "client_weight_index": client_weight_index,
        "client_bp_level": client_bp_level,
        "client_lipid": client_lipid,
        "client_sugar": client_sugar,
        "health_target": health_target,
    }
)

# Inserting Blank Values
raw_df.loc[
    np.random.choice([True, False], size=total_records, p=[0.07, 0.93]), "client_age"
] = np.nan
raw_df.loc[
    np.random.choice([True, False], size=total_records, p=[0.05, 0.95]), "client_sex"
] = np.nan
raw_df.loc[
    np.random.choice([True, False], size=total_records, p=[0.06, 0.94]), "client_zone"
] = np.nan
raw_df.loc[
    np.random.choice([True, False], size=total_records, p=[0.09, 0.91]), "client_weight_index"
] = np.nan
raw_df.loc[
    np.random.choice([True, False], size=total_records, p=[0.08, 0.92]),
    "client_lipid",
] = np.nan
raw_df.loc[
    np.random.choice([True, False], size=total_records, p=[0.07, 0.93]), "client_sugar"
] = np.nan

# Inserting Extreme Spike Outliers
raw_df.loc[np.random.choice(raw_df.index, size=5, replace=False), "client_weight_index"] = np.random.uniform(
    58.0, 72.0, size=5
)
raw_df.loc[np.random.choice(raw_df.index, size=8, replace=False), "client_bp_level"] = (
    np.random.uniform(185.0, 235.0, size=8)
)
raw_df.loc[
    np.random.choice(raw_df.index, size=5, replace=False), "client_lipid"
] = np.random.uniform(380.0, 480.0, size=5)
raw_df.loc[np.random.choice(raw_df.index, size=8, replace=False), "client_sugar"] = (
    np.random.uniform(290.0, 440.0, size=8)
)

raw_df.to_csv("client_fitness_data.csv", index=False)
print("Raw data generated successfully!")

Raw data generated successfully!


### Part A: Handling Missing Values
#### 1. Identify missing values and provide a summary report (percentage per column)
Explanation: In this task, we check the dataset for any missing values (NaN) across all columns and calculate their percentage to understand which columns lack data


In [2]:
import numpy as np
import pandas as pd

raw_df = pd.read_csv("client_fitness_data.csv")

# Calculate missing values count and percentage
blank_count = raw_df.isnull().sum()
blank_percentage = (raw_df.isnull().sum() / len(raw_df)) * 100

# Create Summary Report DataFrame
blank_summary = pd.DataFrame(
    {"Missing Count": blank_count, "Missing Percentage (%)": blank_percentage}
)

# Sort by missing percentage descending
blank_summary = blank_summary.sort_values(
    by="Missing Percentage (%)", ascending=False
)

print("--- Part A (Task 1): Missing Values Summary Report ---")
print(blank_summary)

--- Part A (Task 1): Missing Values Summary Report ---
                     Missing Count  Missing Percentage (%)
client_weight_index             45                     9.0
client_sugar                    39                     7.8
client_age                      35                     7.0
client_lipid                    34                     6.8
client_zone                     30                     6.0
client_sex                      28                     5.6
client_code                      0                     0.0
client_bp_level                  0                     0.0
health_target                    0                     0.0


#### 2. Apply the following imputation techniques and compare results
#### **( 1 ) Simple Imputer (Numerical):** Replace missing weight index with mean or median


In [13]:
from sklearn.impute import SimpleImputer

# Median no use kariye chie kar ke extreme values thi median vadhare robust hoy che
num_filler = SimpleImputer(strategy="median")
raw_df[["client_weight_index"]] = num_filler.fit_transform(raw_df[["client_weight_index"]])
print("Simple Imputer (Numerical - Weight Index) applied successfully!")

Simple Imputer (Numerical - Weight Index) applied successfully!


#### **( 2 ) Simple Imputer (Categorical):** Replace missing Zone with the most frequent value
Answer : For the categorical column client_zone, missing values are replaced with the most frequently occurring value (mode).

In [17]:
zone_filler = SimpleImputer(strategy="most_frequent")
raw_df["client_zone"] = zone_filler.fit_transform(raw_df[["client_zone"]]).ravel()
print("Simple Imputer (Categorical - Zone) applied successfully!")

Simple Imputer (Categorical - Zone) applied successfully!


#### **( 3 ) Most Frequent Imputation:** Replace missing Sex with the most common category
Answer : The client_sex column missing entries are filled using the most common category strategy.

In [18]:
sex_filler = SimpleImputer(strategy="most_frequent")
raw_df["client_sex"] = pd.Series(
    sex_filler.fit_transform(raw_df[["client_sex"]]).ravel(), index=raw_df.index
)
print("Most Frequent Imputation (Sex) applied successfully!")

Most Frequent Imputation (Sex) applied successfully!


#### **( 4 ) Missing Indicator + Random Sample Imputation:** Create binary indicator columns for missingness and use random sampling to impute values
Answer : This technique creates an extra binary column (1 if missing, 0 if present) to retain missingness information, and fills the missing cells using random sampling from existing non-null values

In [19]:
# Applying Missing Indicator + Random Sample Imputation for client_age and client_lipid
for col in ["client_age", "client_lipid"]:
  # 1. Create a binary missing indicator column
  raw_df[f"{col}_missing_indicator"] = raw_df[col].isnull().astype(int)

  # 2. Random sample imputation for missing values
  not_null_values = raw_df[col].dropna()
  missing_mask = raw_df[col].isnull()
  raw_df.loc[missing_mask, col] = np.random.choice(
      not_null_values, size=missing_mask.sum()
  )

print(
    "Missing Indicator + Random Sample Imputation applied successfully!"
)

Missing Indicator + Random Sample Imputation applied successfully!


#### **( 5 ) KNN Imputer:** Apply k-Nearest Neighbors imputation for multivariate imputation
Answer : KNN Imputer finds the nearest neighboring clients based on feature similarity and imputes missing values using their average values.

In [ ]:
from sklearn.impute import KNNImputer

# Select numerical columns for KNN imputation
knn_cols = ["client_age", "client_weight_index", "client_bp_level", "client_lipid", "client_sugar"]
knn_imputer = KNNImputer(n_neighbors=5)
raw_df[knn_cols] = knn_imputer.fit_transform(raw_df[knn_cols])
print("KNN Imputer applied successfully!")

KNN Imputer applied successfully!


#### **( 6 ) MICE Algorithm:** Perform chained equations imputation for multiple variables simultaneously
Answer : MICE (Multivariate Imputation by Chained Equations) models each feature with missing values as a function of other features in a round-robin fashion.

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_cols = ["client_age", "client_weight_index", "client_bp_level", "client_lipid", "client_sugar"]

# Apply MICE Algorithm
mice_imputer = IterativeImputer(max_iter=10, random_state=42)
raw_df[mice_cols] = mice_imputer.fit_transform(raw_df[mice_cols])

print("MICE Algorithm applied successfully!")

MICE Algorithm applied successfully!


  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.28.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\91635\\AppData\\Local\\Programs\\Python\\Python314\\Lib\\site-packages\\sklearn\\_cyutility.cp314-win_amd64.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Part B: Handling Outliers

#### 3. Detect and remove outliers using
##### **-> Z-score method** (For Lipid and Sugar)
Explanation: Z-score measures how many standard deviations a data point is from the mean. Values with an absolute Z-score greater than 3 are typically considered extreme outliers and are removed.

In [22]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# Make a copy of dataframe for Z-score removal
df_zscore = raw_df.copy()
print(f"Shape before Z-score removal: {df_zscore.shape}")

# Calculate Z-scores for lipid and sugar
filtered_index = (
    np.abs(
        stats.zscore(df_zscore[["client_lipid", "client_sugar"]].fillna(0))
    )
    < 3
).all(axis=1)
df_zscore = df_zscore[filtered_index]

print(f"Shape after Z-score removal: {df_zscore.shape}")

Shape before Z-score removal: (500, 11)
Shape after Z-score removal: (486, 11)


##### **-> IQR method** (For Weight Index)
Explanation: The Interquartile Range (IQR) method uses the 25th (Q1) and 75th (Q3) percentiles. Values below $Q1 - 1.5 \\times IQR$ or above $Q3 + 1.5 \\times IQR$ are classified as outliers and removed.

In [23]:
# Make a copy of dataframe for IQR removal
df_iqr = raw_df.copy()
print(f"Shape before IQR removal: {df_iqr.shape}")

# Calculate Q1, Q3, and IQR for weight index
Q1 = df_iqr["client_weight_index"].quantile(0.25)
Q3 = df_iqr["client_weight_index"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out weight index outliers
df_iqr = df_iqr[
    (df_iqr["client_weight_index"] >= lower_bound) & (df_iqr["client_weight_index"] <= upper_bound)
]

print(f"Shape after IQR removal: {df_iqr.shape}")

Shape before IQR removal: (500, 11)
Shape after IQR removal: (489, 11)


##### **-> Percentile method** (Capping below 1st and above 99th percentile)
Explanation: Instead of dropping entire rows, the percentile method caps (clips) extreme values by setting anything below the 1st percentile to the 1st percentile value, and anything above the 99th percentile to the 99th percentile value.

In [24]:
# Make a copy of dataframe for Percentile capping
df_percentile = raw_df.copy()

# Apply percentile capping on continuous features
for col in ["client_weight_index", "client_lipid", "client_sugar"]:
  p1 = df_percentile[col].quantile(0.01)
  p99 = df_percentile[col].quantile(0.99)
  df_percentile[col] = np.clip(df_percentile[col], p1, p99)

print("Percentile capping applied successfully to Weight Index, Lipid, and Sugar!")

Percentile capping applied successfully to Weight Index, Lipid, and Sugar!


#### 4. Apply Winsorization to cap extreme outliers instead of removing them
Explanation: Winsorization replaces extreme outliers with specific percentiles (like 1st and 99th) instead of deleting rows, which helps preserve the sample size


In [25]:
from scipy.stats import mstats

# Applying Winsorization to numerical columns to cap extreme values
cols_to_winsorize = ["client_weight_index", "client_bp_level", "client_lipid", "client_sugar"]

for col in cols_to_winsorize:
  # Limits set to 1% on both lower and upper tails
  raw_df[col] = mstats.winsorize(raw_df[col], limits=[0.01, 0.01])

print("Winsorization applied successfully to all continuous columns!")

Winsorization applied successfully to all continuous columns!


#### 5. Compare dataset shape and summary before vs after outlier treatment

In [28]:
# Comparison of Dataset Shapes and Descriptive Statistics
print("--- Before Outlier Treatment Summary ---")
print(raw_df[["client_weight_index", "client_bp_level", "client_lipid", "client_sugar"]].describe())

# Applying winsorization/cleaning to main dataframe
for col in ["client_weight_index", "client_bp_level", "client_lipid", "client_sugar"]:
  raw_df[col] = mstats.winsorize(raw_df[col], limits=[0.01, 0.01])

print("\n--- After Outlier Treatment Summary ---")
print(raw_df[["client_weight_index", "client_bp_level", "client_lipid", "client_sugar"]].describe())

--- Before Outlier Treatment Summary ---
       client_weight_index  client_bp_level  client_lipid  client_sugar
count           500.000000       500.000000    500.000000    500.000000
mean             25.472750       118.999786    193.493461    100.765517
std               3.902644        15.966951     28.759155     32.896928
min              17.333292        87.898230    133.650244     58.501294
25%              22.851853       108.318533    173.985311     85.673397
50%              25.263011       117.784222    192.670301     97.932831
75%              27.732407       127.099610    212.738633    107.934265
max              36.544383       187.084215    271.074502    332.502963

--- After Outlier Treatment Summary ---
       client_weight_index  client_bp_level  client_lipid  client_sugar
count           500.000000       500.000000    500.000000    500.000000
mean             25.472750       118.999786    193.493461    100.765517
std               3.902644        15.966951     28.759

c:\Users\91635\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\91635\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\91635\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\91635\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\91635\AppData\Local\Programs\Python\Python314\Lib\site-packages\numpy\lib\_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the 

### Part C: Final Clean Dataset & Report

#### 6. Present the final cleaned dataset with

##### -> All missing values treated appropriately.

In [29]:
# Create a copy of the dataframe for treating missing values
df_missing_treated = df.copy()

# Filling missing values for numerical and categorical columns
df_missing_treated["bmi"].fillna(
    df_missing_treated["bmi"].median(), inplace=True
)
df_missing_treated["region"].fillna(
    df_missing_treated["region"].mode()[0], inplace=True
)
df_missing_treated["gender"].fillna(
    df_missing_treated["gender"].mode()[0], inplace=True
)
df_missing_treated["age"].fillna(
    df_missing_treated["age"].median(), inplace=True
)

NameError: name 'df' is not defined

##### -> All outliers handled using suitable methods.

In [28]:
from scipy.stats import mstats

# Use the dataframe where missing values are already treated
df_final_clean = df_missing_treated.copy()

# Handling outliers using Winsorization (Capping at 1st and 99th percentiles)
for col in ["bmi", "blood_pressure", "cholesterol", "glucose"]:
  df_final_clean[col] = mstats.winsorize(df_final_clean[col], limits=[0.01, 0.01])

# Save the final clean dataset to CSV
df_final_clean.to_csv("cleaned_patient_health_records.csv", index=False)

print("--- Sub-point 2: Outliers Handled Successfully ---")
print(df_final_clean.head())
print(
    "\nFinal clean dataset successfully saved as 'cleaned_patient_health_records.csv'!"
)

--- Sub-point 2: Outliers Handled Successfully ---
  patient_id   age  gender region        bmi  blood_pressure  cholesterol  \
0      P0001  56.0  Female  South  21.588743      117.996302   211.384462   
1      P0002  69.0    Male  South  26.503146      136.287995   254.345144   
2      P0003  46.0  Female   West  33.293971       88.486217   186.137119   
3      P0004  32.0    Male  South  28.309155      164.066293   175.129492   
4      P0005  60.0    Male  South  24.936130      106.932994   162.009785   

      glucose  disease_risk  age_missing_indicator  \
0  103.845860             1                      0   
1   97.035542             1                      0   
2   82.430343             1                      0   
3   71.749285             1                      0   
4   99.425820             1                      0   

   cholesterol_missing_indicator  
0                              0  
1                              0  
2                              1  
3                    

#### 7. Provide a brief report explaining


# Data Cleanser: Project Summary Report

1. Most Effective Imputation Strategy:

* **KNN Imputer and MICE Algorithm** proved optimal for numerical attributes (`bmi`, `cholesterol`, `glucose`) because they maintain multi-variable relationships among patient records rather than substituting blank spots blindly with simple averages[cite: 3].
* For categorical fields (`region`, `gender`), **Most Frequent (Mode) Imputation** successfully preserved original categorical ratios[cite: 3].

2. Best Outlier Handling Method:

* **Winsorization (Percentile Capping at 1st and 99th percentiles)** maintained data integrity superiorly[cite: 3]. Contrary to conventional pruning approaches (like Z-score or IQR) that discard entire patient entries and lose critical medical insights, Winsorization successfully dampened extreme data spikes while keeping total observation counts completely untouched for subsequent machine learning operations[cite: 3].

3. Impact of Data Cleaning on Dataset Usability:

* Fixing empty cells and subduing aggressive synthetic outliers eradicated algorithmic skew, prevented distorted spreads, and guaranteed feature reliability[cite: 3].
* The perfected pristine database (`cleaned_patient_health_records.csv`) stands fully normalized, sturdy, and ideal for constructing advanced forecasting models (such as estimating `disease_risk`).